In [ ]:
import json

#Ouvrir le fichier json contenant nos test
try:
    with open("test.json", "r", encoding="utf-8") as f:
        data = json.load(f)
except:
    raise FileExistsError



### Fonctions utiles

In [ ]:
def chunker(texts, chunk_size):
    """
        Fonction qui transforme en chunck de taille variable, un texte ou un liste de text

        @texts: Le texte ou la liste de textes à chunker
        @chunk_size: La taille que doivent respecter les chunks

        @return
            Retourne une liste de chunks
    """
    if isinstance(texts, list):
        chunks = []
        for i, text in enumerate(texts):
            chunks.append([text[i:i+chunk_size] for i in range(0, len(text), chunk_size)])
    else:
        chunks = [texts[i:i+chunk_size] for i in range(0, len(texts), chunk_size)]

    return chunks


def get_text_embedding(input, client):
    embeddings_batch_response = client.embeddings.create(
        model = "mistral-embed",
        inputs = input
    )
    return embeddings_batch_response.data[0].embedding

### Chucker les pharases

In [ ]:
chunks = {}
for elt, items in data.items():

    chunks[elt] = {
        "similary": [],
        "dimissilarity": []
    }
    for relatedness, text in items.items():
        if relatedness == "similarity":
            similarity = text
            chunks[elt]["similary"] = chunker(text, 2048)
               
        else:
            dimissilarity = text
            chunks[elt]["dimissilarity"] = chunker(text, 2048)


with open("chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=4, ensure_ascii=False)

### Embedding avec ***Mistral embeddings***

In [ ]:
from mistralai.client import Mistral
import requests
import numpy as np
import faiss
import os
from dotenv import load_dotenv

load_dotenv() #cHARGER LE FICHIER .ENV automatiquement

#Récupérer la clé d'API
api_key = os.getenv("MISTRAL_API_KEY")


#Initialisation du client Mistral
client = Mistral(api_key=api_key)

embedding = {}
i = 1
for elt, category in chunks.items():#Hauteur catgéories
    print(i)
    embedding[elt] = {}
    for relatedness, sentences in category.items(): #Hauteur similaires et non similaires
        print(relatedness)
        embedding[elt][relatedness] = {}
        embedding[elt][relatedness] = {}

        p=1
        for sentence in sentences: #Hauteur phrases
            text_embedding =  np.array([get_text_embedding(chunk, client=client) for chunk in sentence])
            embedding[elt][relatedness][f"vec_phr_{p}"] = text_embedding.tolist()
            print(text_embedding)
            p+=1
    i+=1
    print("\n\n")
with open("embedding_mistral.json", "w", encoding="utf-8") as f:
    json.dump(embedding, f, indent=4, ensure_ascii=False)
        

    


1
similary
[[-0.04156494  0.0524292   0.05471802 ...  0.00629044  0.02328491
  -0.02839661]]
[[-0.02998352  0.00531769  0.05551147 ... -0.0297699  -0.00359535
   0.01094055]]
dimissilarity
[[-0.01222992  0.01445007  0.03131104 ... -0.01596069  0.0085907
   0.03131104]]



2
similary
[[-0.01222992  0.01445007  0.03131104 ... -0.01596069  0.0085907
   0.03131104]]
[[-0.01347351  0.0073967   0.05471802 ... -0.01296997  0.00794983
   0.01550293]]
dimissilarity
[[-0.01812744  0.02493286  0.03933716 ... -0.02223206  0.00963593
  -0.00141335]]



3
similary
[[-0.00924683  0.00368881  0.07501221 ...  0.0037384   0.02203369
  -0.01534271]]
[[-0.04214478  0.01573181  0.03063965 ... -0.0085144  -0.01278687
  -0.01846313]]
dimissilarity
[[-0.03607178  0.00932312  0.05871582 ... -0.01916504 -0.01986694
  -0.01559448]]



4
similary
[[-0.01402283  0.04718018  0.06256104 ... -0.02023315  0.01449585
  -0.01186371]]
[[-0.03207397  0.06228638  0.04559326 ... -0.01576233  0.02832031
  -0.01885986]]
dimis

### Embedding avec ***SBERT***

In [ ]:
from sentence_transformers import SentenceTransformer

#1 Charger le modèle préentrainé Sentence Transformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

#Mettre toutes les phrases ensemble dans un liste en fonction de la section docs
embedding = {}
for elt, category in data.items():#Hauteur catgéories
    embedding[elt] = []
    for relatedness, sentences in category.items(): #Hauteur similaires et non similaires
        for sentence in sentences: #Hauteur phrases
            embedding[elt].append(sentence)

text_embedded = {}
for category, text in embedding.items():
    text_embedded[category] = (model.encode(text)).tolist()

with open("embedding_SBERT.json", "w", encoding="utf-8") as f:
    json.dump(text_embedded, f, indent=4, ensure_ascii=False)

/Users/cheick/Documents/repository/trifouillis-city-hal/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7406.75it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Embedding avec ***FastText***

In [ ]:
import fasttext
import numpy as np

#Téléchartger un modèle pré-entrainé
model = fasttext.load_model('cc.en.300.bin')

#Mettre toutes les phrases ensemble dans un liste en fonction de la section docs
embedding = {}
for elt, category in data.items():#Hauteur catgéories
    embedding[elt] = []
    for relatedness, sentences in category.items(): #Hauteur similaires et non similaires
        for sentence in sentences: #Hauteur phrases
            embedding[elt].append(sentence)

text_embedded = {}
for category, texts in embedding.items():
    text_embedded[category] = [ model.get_sentence_vector(t).tolist() for t in text ]



with open("embedding_fastText.json", "w", encoding="utf-8") as f:
    json.dump(text_embedded, f, indent=4, ensure_ascii=False)

KeyboardInterrupt: 